# Lesson 3: Street Network Analysis

**Planning question: How connected and walkable are the streets?**

The pattern of streets shapes how people move. Small blocks with many intersections make walking easy.
Long blocks and dead ends force long detours.

In this notebook you will compare two very different parts of Bengaluru:

- **Jayanagar**, a planned layout from the 1940s with a grid of streets
- **Chickpet**, an old market area (the "Pete") that grew without a fixed plan

Part of [RS & GIS for Spatial Planners](https://soniadas123.github.io/rs-gis-spatial-planner/).

In [ ]:
!pip install -q osmnx

In [ ]:
import osmnx as ox
import pandas as pd
import matplotlib.pyplot as plt

## Settings

We study a 2 km by 2 km square around each place. `DIST` is half the side of that square.

In [ ]:
PLACES = {
    "Jayanagar": (12.9250, 77.5938),
    "Chickpet": (12.9698, 77.5770),
}
DIST = 1000  # metres

## Step 1: Download the walking networks

A street network is stored as a **graph**: **nodes** are intersections and dead ends, and **edges** are the street segments between them.
`network_type="walk"` keeps the streets and paths that people can walk on.

In [ ]:
graphs = {}
for name, center in PLACES.items():
    graphs[name] = ox.graph_from_point(center, dist=DIST, network_type="walk")
    print(name, "- nodes:", len(graphs[name].nodes), " edges:", len(graphs[name].edges))

## Step 2: Look at the two networks side by side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, (name, G) in zip(axes, graphs.items()):
    ox.plot_graph(G, ax=ax, node_size=0, edge_color="black", edge_linewidth=0.6,
                  bgcolor="white", show=False, close=False)
    ax.set_title(name, fontsize=16)
plt.show()

## Step 3: Measure the networks

`basic_stats` calculates the standard street network measures. We pick the ones planners use most:

| Measure | What it means | For walkability |
|---|---|---|
| Intersection density | Intersections per square km | Higher is better |
| Street density | Km of street per square km | Higher is better |
| Average street segment length | Average block side in metres | Shorter is better |
| Streets per node | Average number of streets meeting at a junction | Higher is better |
| Dead-end share | Share of nodes that are dead ends | Lower is better |
| Circuity | Street distance divided by straight-line distance | Closer to 1 is better |

The graph must be **projected** to metres before measuring.

In [ ]:
area_m2 = (2 * DIST) ** 2
rows = []
for name, G in graphs.items():
    G_proj = ox.project_graph(G)
    s = ox.stats.basic_stats(G_proj, area=area_m2)
    rows.append({
        "Place": name,
        "Intersection density (per sq km)": round(s["intersection_density_km"]),
        "Street density (km per sq km)": round(s["street_density_km"] / 1000, 1),
        "Avg segment length (m)": round(s["street_length_avg"]),
        "Streets per node": round(s["streets_per_node_avg"], 2),
        "Dead-end share (%)": round(s["streets_per_node_proportions"].get(1, 0) * 100, 1),
        "Circuity": round(s["circuity_avg"], 3),
    })

comparison = pd.DataFrame(rows).set_index("Place")
comparison

## Step 4: Where are the dead ends?

Each node has a `street_count`, the number of streets that meet there.
A count of 1 is a dead end, and 3 or more is a real junction. Let us map them.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, (name, G) in zip(axes, graphs.items()):
    nodes, edges = ox.graph_to_gdfs(G)
    edges.plot(ax=ax, color="lightgrey", linewidth=0.6)
    nodes[nodes["street_count"] >= 3].plot(ax=ax, color="teal", markersize=4, label="Junction")
    nodes[nodes["street_count"] == 1].plot(ax=ax, color="red", markersize=8, label="Dead end")
    ax.set_title(name, fontsize=16)
    ax.legend(loc="lower left")
    ax.set_axis_off()
plt.show()

## Step 5: Street orientation

Which directions do the streets run in? A **polar histogram** shows how many streets point in each compass direction.
A grid gives a cross shape. An organic pattern spreads in all directions.

**Orientation entropy** sums this up in one number. Lower means more ordered, like a grid.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6), subplot_kw={"projection": "polar"})
for ax, (name, G) in zip(axes, graphs.items()):
    G_undirected = ox.convert.to_undirected(G)
    entropy = ox.bearing.orientation_entropy(G_undirected)
    ox.plot.plot_orientation(G_undirected, ax=ax, title=f"{name}\nentropy = {entropy:.2f}", area=True)
plt.tight_layout()
plt.show()

## Step 6: Save the networks

Each network is saved as a GeoPackage with two layers, nodes and edges, which you can open in QGIS.

In [ ]:
for name, G in graphs.items():
    ox.save_graph_geopackage(G, filepath=f"{name.lower()}_walk_network.gpkg")
print("Saved")

## What this means for planning

- Planned grids such as Jayanagar usually have regular blocks, few dead ends and a clear orientation.
- Old cores such as Chickpet can have very high intersection density. They are walkable, but hard to navigate and to serve with vehicles.
- These measures help you to compare a proposed layout with existing ones, and to argue for new links in gated or dead-end layouts.

## Try it yourself

1. Add a newer area such as Whitefield `(12.9698, 77.7500)` to `PLACES`. How does a recent suburb compare?
2. Change `network_type` to `"drive"`. What changes in the measures?
3. Increase `DIST` to 1500. Do the measures stay similar?

---
*Enjoying the course? Support it with a [star on GitHub](https://github.com/soniadas123/rs-gis-spatial-planner).*